# 02 — Automated EDA PDF Report

This notebook loads a CSV (or uses a synthetic fallback), performs a compact EDA, and
**exports a multi-page PDF report** to `artifacts/eda_report.pdf` using `matplotlib.backends.backend_pdf.PdfPages`.

**What it includes**
- Dataset overview (shape, dtypes, missingness)
- Numeric histograms & boxplots
- Categorical top-k bar charts
- Pearson correlation heatmap (numeric)
- Cramér's V heatmap (categorical)
- Optional target analysis (if `TARGET_COLUMN` is set)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from typing import List, Optional
from scipy.stats import chi2_contingency

ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

# ==== Configure your dataset here ===============================================
CSV_PATH = "your_dataset.csv"   # e.g., "../data/your_file.csv"
TARGET_COLUMN = None            # e.g., "price" (set to None if unknown)
TOP_K_CATS = 20                 # top-k categories to display per categorical feature

# ==== Load data (with synthetic fallback) =======================================
if not Path(CSV_PATH).exists():
    rng = np.random.default_rng(42)
    n = 800
    df = pd.DataFrame({
        "age": rng.normal(40, 12, n).round(0),
        "income": rng.lognormal(mean=10, sigma=0.5, size=n),
        "city": rng.choice(["Rome", "Milan", "Naples", "Turin"], size=n, replace=True),
        "is_premium": rng.choice([0,1], size=n, replace=True, p=[0.7,0.3]),
    })
    df["spend"] = 200 + 0.8*df["income"] + 3.0*df["age"] + rng.normal(0, 200, n)
    TARGET_COLUMN = "spend"
    print("Loaded synthetic dataset — set CSV_PATH to your file to use a real dataset.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded: {CSV_PATH}")

# ==== Helpers ===================================================================
def split_feature_types(df: pd.DataFrame, target: Optional[str]=None):
    cats, nums = [], []
    for c in df.columns:
        if target is not None and c == target:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            nums.append(c)
        else:
            cats.append(c)
    return cats, nums

def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    m = df.isna().sum().sort_values(ascending=False)
    pct = (m / len(df)).round(4)
    out = pd.DataFrame({"missing": m, "missing_pct": pct})
    return out[out["missing"] > 0]

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    tbl = pd.crosstab(x, y)
    chi2, p, dof, ex = chi2_contingency(tbl, correction=False)
    n = tbl.values.sum()
    return float(np.sqrt((chi2 / n) / (min(tbl.shape) - 1)))

# ==== Build PDF report ===========================================================
pdf_path = ARTIFACTS / "eda_report.pdf"
with PdfPages(pdf_path) as pdf:
    # Cover / overview
    fig = plt.figure()
    plt.axis("off")
    title = "EDA Report"
    info = [
        f"Rows, Columns: {df.shape}",
        "",
        "Dtypes:",
        df.dtypes.to_string(),
        "",
        "Missing values:",
        (missing_summary(df).to_string() if not missing_summary(df).empty else "None"),
    ]
    plt.text(0.0, 1.0, title, fontsize=18, va="top", ha="left")
    plt.text(0.0, 0.9, "\n".join(info), fontsize=9, va="top", ha="left", family="monospace")
    pdf.savefig(fig); plt.close(fig)

    cats, nums = split_feature_types(df, TARGET_COLUMN)

    # Numeric distributions
    for col in nums:
        s = df[col].dropna()
        if s.empty: 
            continue

        fig = plt.figure()
        plt.hist(s, bins=30)
        plt.title(f"Histogram — {col}")
        plt.xlabel(col); plt.ylabel("Count")
        pdf.savefig(fig); plt.close(fig)

        fig = plt.figure()
        plt.boxplot(s, vert=True, labels=[col])
        plt.title(f"Boxplot — {col}")
        pdf.savefig(fig); plt.close(fig)

    # Categorical distributions
    for col in cats:
        vc = df[col].astype("object").value_counts(dropna=False).head(TOP_K_CATS)
        fig = plt.figure()
        plt.bar(vc.index.astype(str), vc.values)
        plt.title(f"Top categories — {col}")
        plt.xticks(rotation=45, ha="right"); plt.ylabel("Count")
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

    # Numeric correlation heatmap
    if len(nums) >= 2:
        corr = df[nums].corr(numeric_only=True).fillna(0.0)
        fig = plt.figure(figsize=(6,5))
        plt.imshow(corr.values, interpolation="nearest")
        plt.xticks(range(len(nums)), nums, rotation=45, ha="right")
        plt.yticks(range(len(nums)), nums)
        plt.title("Pearson Correlation (numeric)"); plt.colorbar()
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

    # Cramér's V heatmap for categoricals
    if len(cats) >= 2:
        import numpy as np
        cv_mat = np.zeros((len(cats), len(cats)))
        for i, ci in enumerate(cats):
            for j, cj in enumerate(cats):
                if i == j:
                    cv_mat[i, j] = 1.0
                elif i < j:
                    try:
                        cv = cramers_v(df[ci].astype("object"), df[cj].astype("object"))
                    except Exception:
                        cv = np.nan
                    cv_mat[i, j] = cv_mat[j, i] = cv
        fig = plt.figure(figsize=(6,5))
        plt.imshow(cv_mat, interpolation="nearest")
        plt.xticks(range(len(cats)), cats, rotation=45, ha="right")
        plt.yticks(range(len(cats)), cats)
        plt.title("Cramér's V (categorical)"); plt.colorbar()
        plt.tight_layout()
        pdf.savefig(fig); plt.close(fig)

    # Target analysis page
    if TARGET_COLUMN is not None and TARGET_COLUMN in df.columns:
        y = df[TARGET_COLUMN]
        is_regression = pd.api.types.is_numeric_dtype(y) and y.nunique() > 15
        fig = plt.figure()
        plt.axis("off")
        plt.text(0.0, 1.0, f"Target Analysis — {TARGET_COLUMN}", fontsize=14, va="top", ha="left")
        details = [
            f"Type: {'Regression' if is_regression else 'Classification'}",
            f"Nulls: {df[TARGET_COLUMN].isna().sum()}",
            f"Unique: {df[TARGET_COLUMN].nunique()}",
        ]
        if is_regression:
            details += [
                f"Mean: {y.mean():.3f}",
                f"Std : {y.std():.3f}",
                f"Min : {y.min():.3f}",
                f"Max : {y.max():.3f}",
            ]
        else:
            details += [
                "Class counts:",
                y.value_counts(dropna=False).to_string()
            ]
        plt.text(0.0, 0.9, "\n".join(details), fontsize=10, va="top", ha="left", family="monospace")
        pdf.savefig(fig); plt.close(fig)

print("Saved PDF report to:", pdf_path.resolve())